# Reranker Experiment (Cross-Encoder)

This standalone notebook runs cross-encoder re-ranking experiments using benchmark datasets from CoIR (Code Information Retrieval), evaluates on English and Indonesian queries, and displays results. 

It operates independently without needing to clone a GitHub repository, downloading datasets directly from HuggingFace.

## 1. Setup Environment
Install the required libraries.

In [ ]:
!pip install -q transformers sentence-transformers datasets numpy pandas tqdm huggingface_hub

## 2. Imports & Setup

In [ ]:
import gc
import json
import logging
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from typing import List, Dict, Any, Optional
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


## 3. Define Retriever and Reranker Components
Here we define the `DenseRetriever` for first-stage retrieval and the `CrossEncoderReranker` for re-scoring candidates.

In [ ]:
class DenseRetriever:
    QUERY_PREFIX = "query: "
    PASSAGE_PREFIX = "passage: "
    
    def __init__(
        self,
        model_name: str = "intfloat/multilingual-e5-small",
        device: Optional[str] = None,
        batch_size: int = 32,
        normalize_embeddings: bool = True,
    ):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.normalize_embeddings = normalize_embeddings
        
        logger.info(f"Loading retriever model: {model_name} on {self.device}")
        self.model = SentenceTransformer(model_name)
        self.model.to(self.device)
        self.corpus_embeddings = None
        self.corpus_ids = None

    def encode_queries(self, queries: List[str]) -> np.ndarray:
        prefixed_queries = [self.QUERY_PREFIX + q for q in queries]
        return self.model.encode(
            prefixed_queries,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device=self.device,
            normalize_embeddings=self.normalize_embeddings,
        )

    def encode_corpus(self, corpus: List[Dict[str, str]]):
        texts = [doc.get("text", "") for doc in corpus]
        doc_ids = [doc.get("id", str(i)) for i, doc in enumerate(corpus)]
        prefixed_texts = [self.PASSAGE_PREFIX + t for t in texts]
        
        embeddings = self.model.encode(
            prefixed_texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device=self.device,
            normalize_embeddings=self.normalize_embeddings,
        )
        self.corpus_embeddings = embeddings
        self.corpus_ids = doc_ids
        return embeddings, doc_ids

    def retrieve(self, queries: List[str], corpus: List[Dict[str, str]], top_k: int = 100):
        query_embeddings = self.encode_queries(queries)
        
        if self.corpus_embeddings is None:
            corpus_embeddings, doc_ids = self.encode_corpus(corpus)
        else:
            corpus_embeddings = self.corpus_embeddings
            doc_ids = self.corpus_ids
            
        similarities = np.matmul(query_embeddings, corpus_embeddings.T)
        
        results = []
        for i in range(len(queries)):
            scores = similarities[i]
            top_indices = np.argsort(scores)[-top_k:][::-1]
            result = [
                {"id": doc_ids[idx], "score": float(scores[idx]), "rank": r + 1}
                for r, idx in enumerate(top_indices)
            ]
            results.append(result)
        return results


class StandaloneCrossEncoderReranker:
    def __init__(
        self, 
        model_name: str, 
        device: Optional[str] = None, 
        max_length: int = 512, 
        batch_size: int = 8
    ):
        self.model_name = model_name
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.max_length = max_length
        self.batch_size = batch_size
        self.model = None
        self.tokenizer = None
        
    def load_model(self):
        logger.info(f"Loading reranker model: {self.model_name} on {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name).to(self.device)
        
    def score(self, query: str, documents: List[str]) -> List[float]:
        if self.model is None:
            self.load_model()
            
        pairs = [[query, doc] for doc in documents]
        inputs = self.tokenizer(
            pairs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        ).to(self.device)
        
        scores = []
        with torch.no_grad():
            for i in range(0, len(pairs), self.batch_size):
                batch_inputs = {k: v[i:i+self.batch_size] for k, v in inputs.items()}
                outputs = self.model(**batch_inputs)
                batch_scores = outputs.logits.squeeze(-1).cpu().numpy()
                
                # Handle scalar case if batch size is 1
                if batch_scores.ndim == 0:
                    scores.append(float(batch_scores))
                else:
                    scores.extend(batch_scores.tolist())
                    
        return scores


## 4. Load Dataset
Fetches the COSQA benchmark directly from HuggingFace (`CoIR-Retrieval`). Optionally mounts external Indonesian translations.

In [ ]:
def load_cosqa_data():
    # Optional: If you uploaded translations CSV to the colab environment, it will be loaded here.
    translations_file = "cosqa_queries_indonesian.csv"
    if Path(translations_file).exists():
        trans_df = pd.read_csv(translations_file, sep="|")
        translations = dict(zip(trans_df['qid'], trans_df['query_id']))
    else:
        logger.warning(f"{translations_file} not found. Using English dataset without translations.")
        translations = {}
        
    logger.info("Loading COSQA from HuggingFace datasets...")
    queries_corpus_dataset = load_dataset("CoIR-Retrieval/cosqa-queries-corpus")
    qrels_dataset = load_dataset("CoIR-Retrieval/cosqa-qrels")
    
    # Process corpus
    corpus_data = queries_corpus_dataset['corpus']
    corpus = [{"id": str(item["_id"]), "text": item.get("text", "")} for item in corpus_data]
    
    # Process queries & qrels
    query_data = queries_corpus_dataset['queries']
    queries_en = {str(item["_id"]): item.get("text", "") for item in query_data}
    
    qrels_data = qrels_dataset['test']
    qrels = {}
    for item in qrels_data:
        qid = str(item['query_id'])
        doc_id = str(item['corpus_id'])
        score = int(item['score'])
        if qid not in qrels: qrels[qid] = {}
        qrels[qid][doc_id] = score
        
    # Process indonesian queries
    queries_id = {}
    for qid, qtext in queries_en.items():
        queries_id[qid] = translations.get(qid, qtext)
        
    return corpus, queries_en, queries_id, qrels

corpus, queries_en, queries_id, qrels = load_cosqa_data()

# Subsample for testing to keep execution time low (optional, set to None for full evaluation)
SAMPLE_SIZE = 100 
if SAMPLE_SIZE:
    queries_en = dict(list(queries_en.items())[:SAMPLE_SIZE])
    queries_id = dict(list(queries_id.items())[:SAMPLE_SIZE])
    qrels = {k: v for k, v in qrels.items() if k in queries_en}
    print(f"Sampled to {SAMPLE_SIZE} queries for quick testing.")

print(f"Loaded {len(queries_en)} queries, {len(corpus)} docs")


## 5. First-Stage Retrieval (mE5)
Use multilingual-e5-small to retrieve the top 100 documents for each query.

In [ ]:
def run_first_stage_retrieval(queries: dict, corpus: list, top_k: int = 100):
    logger.info("Running first-stage retrieval with mE5...")
    retriever = DenseRetriever(
        model_name="intfloat/multilingual-e5-small",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    
    corpus_lookup = {doc['id']: doc for doc in corpus}
    queries_list = list(queries.values())
    qids_list = list(queries.keys())
    
    retrieved_all = retriever.retrieve(queries_list, corpus, top_k=top_k)
    results = {}
    
    for qid, query, retrieved in zip(qids_list, queries_list, retrieved_all):
        results_with_text = []
        for doc in retrieved:
            doc_id = doc['id']
            if doc_id in corpus_lookup:
                results_with_text.append({
                    **doc,
                    'text': corpus_lookup[doc_id].get('text', '')
                })
            else:
                results_with_text.append(doc)
        results[qid] = {
            "query": query,
            "retrieved": results_with_text
        }
        
    return results

FIRST_STAGE_K = 100
first_stage_en = run_first_stage_retrieval(queries_en, corpus, FIRST_STAGE_K)
first_stage_id = run_first_stage_retrieval(queries_id, corpus, FIRST_STAGE_K)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. Re-Ranking
Apply the selected cross-encoder model to re-score and re-rank the retrieved documents.

In [ ]:
def run_reranking(first_stage_results: dict, reranker, top_k: int = 10):
    results = []
    for qid, result in tqdm(first_stage_results.items(), desc="Reranking"):
        query = result["query"]
        first_stage_docs = result["retrieved"]
        
        # Cross encoder allows larger input, truncating individually if necessary
        doc_texts = [doc['text'][:512] for doc in first_stage_docs]
        scores = reranker.score(query, doc_texts)
        
        reranked_docs = []
        for doc, score in zip(first_stage_docs, scores):
            reranked_docs.append({
                **doc,
                'cross_encoder_score': float(score)
            })
        
        reranked_docs.sort(key=lambda x: x['cross_encoder_score'], reverse=True)
        
        results.append({
            "qid": qid,
            "query": query,
            "first_stage": first_stage_docs[:top_k],
            "reranked": reranked_docs[:top_k]
        })
    return results

# Select Reranker Model
MODEL_IDENTIFIER = "castorini/mmarco-mMiniLMv2-L12-H384-uncased"  # equivalent to "mmmini"
# MODEL_IDENTIFIER = "cross-encoder/mmarco-bert-base-multilingual-cased" # equivalent to "mmmini_multilingual"
# MODEL_IDENTIFIER = "xlm-roberta-base" # Note: Needs to be a sequence classification model trained on MSMARCO if you want good scores out of the box

TOP_K = 10

reranker = StandaloneCrossEncoderReranker(
    model_name=MODEL_IDENTIFIER, 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

logger.info("Reranking English queries...")
results_en = run_reranking(first_stage_en, reranker, TOP_K)

logger.info("Reranking Indonesian queries...")
results_id = run_reranking(first_stage_id, reranker, TOP_K)


## 7. Evaluation metrics
Compute NDCG@10 and MAP@10 to measure effectiveness.

In [ ]:
def evaluate_results(results: list, qrels: dict, top_k: int = 10):
    ndcg_scores_before, ndcg_scores_after = [], []
    map_scores_before, map_scores_after = [], []
    
    for result in results:
        qid = result["qid"]
        relevant_docs = qrels.get(qid, {})
        if not relevant_docs:
            continue
        
        relevant_ids = set(relevant_docs.keys())
        
        # Evaluate First Stage (Before)
        before_docs = result.get("first_stage", [])[:top_k]
        before_ids = [doc['id'] for doc in before_docs]
        
        dcg = sum(1.0 / np.log2(i + 2) for i, doc_id in enumerate(before_ids) if doc_id in relevant_ids)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_ids), top_k)))
        ndcg_before = dcg / idcg if idcg > 0 else 0.0
        ndcg_scores_before.append(ndcg_before)
        
        prec_sum = sum(
            sum(1 for doc_id in before_ids[:i+1] if doc_id in relevant_ids) / (i+1)
            for i in range(min(10, len(before_ids)))
            if any(doc_id in relevant_ids for doc_id in before_ids[:i+1])
        )
        map_before = prec_sum / len(relevant_ids) if relevant_ids else 0
        map_scores_before.append(map_before)
        
        # Evaluate Reranked Stage (After)
        after_docs = result.get("reranked", [])[:top_k]
        after_ids = [doc['id'] for doc in after_docs]
        
        dcg = sum(1.0 / np.log2(i + 2) for i, doc_id in enumerate(after_ids) if doc_id in relevant_ids)
        ndcg_after = dcg / idcg if idcg > 0 else 0.0
        ndcg_scores_after.append(ndcg_after)
        
        prec_sum = sum(
            sum(1 for doc_id in after_ids[:i+1] if doc_id in relevant_ids) / (i+1)
            for i in range(min(10, len(after_ids)))
            if any(doc_id in relevant_ids for doc_id in after_ids[:i+1])
        )
        map_after = prec_sum / len(relevant_ids) if relevant_ids else 0
        map_scores_after.append(map_after)
    
    return {
        "ndcg_before": float(np.mean(ndcg_scores_before)) if ndcg_scores_before else 0.0,
        "ndcg_after": float(np.mean(ndcg_scores_after)) if ndcg_scores_after else 0.0,
        "map_before": float(np.mean(map_scores_before)) if map_scores_before else 0.0,
        "map_after": float(np.mean(map_scores_after)) if map_scores_after else 0.0,
        "num_queries": len(results)
    }

metrics_en = evaluate_results(results_en, qrels, TOP_K)
metrics_id = evaluate_results(results_id, qrels, TOP_K)

print("\n" + "=" * 60)
print("RERANKER BENCHMARK RESULTS")
print("=" * 60)
print(f"\nModel: {MODEL_IDENTIFIER}")
print(f"First-stage: mE5 (top-{FIRST_STAGE_K})")
print(f"Top-K: {TOP_K}")

print("\n--- English Queries ---")
print(f"NDCG@{TOP_K} Before: {metrics_en['ndcg_before']:.4f}")
print(f"NDCG@{TOP_K} After:  {metrics_en['ndcg_after']:.4f}")
print(f"MAP@{TOP_K} Before: {metrics_en['map_before']:.4f}")
print(f"MAP@{TOP_K} After:  {metrics_en['map_after']:.4f}")

print("\n--- Indonesian Queries ---")
print(f"NDCG@{TOP_K} Before: {metrics_id['ndcg_before']:.4f}")
print(f"NDCG@{TOP_K} After:  {metrics_id['ndcg_after']:.4f}")
print(f"MAP@{TOP_K} Before: {metrics_id['map_before']:.4f}")
print(f"MAP@{TOP_K} After:  {metrics_id['map_after']:.4f}")

ndcg_imp_en = metrics_en['ndcg_after'] - metrics_en['ndcg_before']
ndcg_imp_id = metrics_id['ndcg_after'] - metrics_id['ndcg_before']
print("\n--- Improvement (After - Before) ---")
print(f"English NDCG: {ndcg_imp_en:+.4f}")
print(f"Indonesian NDCG: {ndcg_imp_id:+.4f}")
print("=" * 60)


## 8. Save results
Export metrics to JSON.

In [ ]:
output_path = Path("reranker_benchmark_results.json")

output_data = {
    "method": MODEL_IDENTIFIER,
    "top_k": TOP_K,
    "first_stage_k": FIRST_STAGE_K,
    "metrics": {
        "english": metrics_en,
        "indonesian": metrics_id
    }
}

with open(output_path, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"Metrics saved to {output_path}")
